In [94]:
import pandas as pd
import os
import shutil
import re
from utils import extract_date

In [95]:

# ── directories ───────────────────────────────────────────
source_dir     = "3_profiling"
cleaning_dir   = "4_cleaned"          # where cleaned CSVs will go
processed_dir  = os.path.join(source_dir, "processed")
meta_data_path = "meta_data.csv"      # master metadata file


In [98]:
class DataCleaning:
    def __init__(self, df_path: str, meta_data_path: str):
        # ── load raw data ──────────────────────────────────
        self.df = pd.read_csv(df_path)

        # Extract table name after the 3‑part date prefix
        # e.g. 4_15_2025_categories.csv ➜ categories.csv
        self.table_name = re.sub(r"^\d+_\d+_\d+_", "", os.path.basename(df_path))

        # ── load metadata and slice just this table ───────
        self.meta_data = pd.read_csv(meta_data_path)
        mask = self.meta_data["table_name"] == self.table_name
        self.meta_data_table = self.meta_data[mask]

    # ─────────────────────────────────────────────────────
    def validate_primary_keys(self):
        """Remove rows with NULLs in PK columns and drop PK duplicates."""
        p_keys = (
            self.meta_data_table[self.meta_data_table["PK"] == 1]["column_name"]
            .tolist()
        )
        if not p_keys:
            print("⚠️  No primary keys defined.")
            return self.df

        # 1) drop rows where any PK col is null
        self.df = self.df[self.df[p_keys].notna().all(axis=1)]

        # 2) drop duplicate composite‑PK rows (keep first)
        self.df = self.df.drop_duplicates(subset=p_keys, keep="first")
        return self

    # ─────────────────────────────────────────────────────
    def validate_non_nulls(self):
        """Remove rows violating NON_NULLABLE constraints."""
        non_null_cols = (
            self.meta_data_table[self.meta_data_table["NON_NULLABLE"] == 1]["column_name"]
            .tolist()
        )
        if non_null_cols:
            self.df = self.df[self.df[non_null_cols].notna().all(axis=1)]
        return self

    # ─────────────────────────────────────────────────────
    def validate_unique(self):
        """Drop duplicate values in columns marked UNIQUE (keep first)."""
        unique_cols = (
            self.meta_data_table[self.meta_data_table["UNIQUE"] == 1]["column_name"]
            .tolist()
        )
        for col in unique_cols:
            self.df = self.df.drop_duplicates(subset=[col], keep="first")
        return self
    # def validate_datatype(self):
    #     dtypes_list = self.meta_data_table['dtype']
    #     columns_list = self.meta_data_table['column_name']
        
    #     key_val = dict(zip(columns_list , dtypes_list))
        
    #     for key, value in key_val.items():
            
    #         pass
    
    def validate_datatype(self):
        """
        Conform each column to the dtype declared in metadata.
        Rows that cannot be coerced are removed so the output DF
        exactly matches the schema.
        """
        # Make sure we’re working on a copy we can mutate safely.
        df = self.df.copy()

        for col, dtype in zip(self.meta_data_table["column_name"],
                            self.meta_data_table["dtype"]):

            if col not in df.columns:
                # Column missing in data — skip or raise, your choice
                print(f"⚠️  Column {col} not found in DataFrame, skipping.")
                continue

            match dtype.lower():
                # ───────────────────────────────────────────── int
                case "int" | "integer":
                    coerced = pd.to_numeric(df[col], errors="coerce", downcast="integer")
                    bad_mask = coerced.isna() & df[col].notna()   # rows that failed coercion
                    if bad_mask.any():
                        df = df[~bad_mask]
                    df[col] = coerced.astype("Int64")             # nullable integer

                # ───────────────────────────────────────────── double / float
                case "double" | "float":
                    coerced = pd.to_numeric(df[col], errors="coerce")
                    bad_mask = coerced.isna() & df[col].notna()
                    if bad_mask.any():
                        df = df[~bad_mask]
                    df[col] = coerced.astype(float)

                # ───────────────────────────────────────────── date / datetime
                case "date" | "datetime":
                    coerced = pd.to_datetime(df[col], errors="coerce", utc=False)
                    bad_mask = coerced.isna() & df[col].notna()
                    if bad_mask.any():
                        df = df[~bad_mask]
                    df[col] = coerced            # keep as pandas datetime

                # ───────────────────────────────────────────── string / other
                case _:
                    # Force to string (object) for any 'string' or un‑mapped dtype
                    df[col] = df[col].astype(str)

        # Save back into the instance and allow chaining
        self.df = df.reset_index(drop=True)
        return self

In [112]:

for filename in os.listdir(source_dir):
    file_path = os.path.join(source_dir, filename)

    if not os.path.isfile(file_path):
        continue   # skip any sub‑directories

    print(f"Processing {filename}")

    # 1) Build /processed/<date>/  and sub‑folders
    date_label  = extract_date(filename)              # e.g. 4_15_2025
    archive_dir = os.path.join(processed_dir, date_label)
    os.makedirs(archive_dir, exist_ok=True)

    # 2) Clean the data
    try:
        cleaner      = DataCleaning(file_path, meta_data_path)
        cleaned_df   = (
            cleaner
            .validate_primary_keys()
            .validate_non_nulls()
            .validate_unique().
            validate_datatype().
            df
        )
        

        
        # 3) Save cleaned CSV into cleaning_dir
        cleaned_path = os.path.join(cleaning_dir, filename)
        cleaned_df.to_csv(cleaned_path, index=False)
        print(f"✓ Cleaned file saved to {cleaned_path}")

    except Exception as e:
        print(f"⚠️  Cleaning failed for {filename}: {e}")
        # Decide here if you want to skip archiving on failure
        continue

    # 4) Move the raw (now‑processed) file to the archive folder
    shutil.move(file_path, os.path.join(archive_dir, filename))
    print(f"→ Raw file archived to {archive_dir}\n")

Processing 4_17_2025_brands.csv
✓ Cleaned file saved to 4_cleaned\4_17_2025_brands.csv
→ Raw file archived to 3_profiling\processed\4_17_2025

Processing 4_17_2025_categories.csv
✓ Cleaned file saved to 4_cleaned\4_17_2025_categories.csv
→ Raw file archived to 3_profiling\processed\4_17_2025

Processing 4_17_2025_customers.csv
✓ Cleaned file saved to 4_cleaned\4_17_2025_customers.csv
→ Raw file archived to 3_profiling\processed\4_17_2025

Processing 4_17_2025_orderitems.csv
✓ Cleaned file saved to 4_cleaned\4_17_2025_orderitems.csv
→ Raw file archived to 3_profiling\processed\4_17_2025

Processing 4_17_2025_orders.csv
✓ Cleaned file saved to 4_cleaned\4_17_2025_orders.csv
→ Raw file archived to 3_profiling\processed\4_17_2025

Processing 4_17_2025_products.csv
✓ Cleaned file saved to 4_cleaned\4_17_2025_products.csv
→ Raw file archived to 3_profiling\processed\4_17_2025

Processing 4_17_2025_staffs.csv
✓ Cleaned file saved to 4_cleaned\4_17_2025_staffs.csv
→ Raw file archived to 3_prof